# Урок 19 — Ітератори та Генератори як Система Потоку Даних

**Модуль 2 · Python Intermediate · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Сьогодні ми перестанемо думати про програмування як про "виконання рядків коду".  
Як архітектори, ми маємо мислити категоріями **потоків даних (Data Flow)**.

Уявіть, що ваша програма — це завод. Дані — це сировина.  
Якщо ви використовуєте класичні структури (списки), ви завозите на завод усю сировину одразу, завалюючи склади (оперативну пам'ять).  
Ми ж навчимося будувати **конвеєри (pipelines)**, де дані обробляються "на льоту", деталь за деталлю, використовуючи принципи **лінивих обчислень (lazy evaluation)**.

| Частина | Патерн | Ключова ідея |
|---------|--------|--------------|
| 1 | Iterator Pattern | Стандарт перебору та внутрішня магія `for` |
| 2 | Generator Pattern | Lazy Execution через `yield` |
| 3 | Infinite Stream Pattern | Нескінченні послідовності без переповнення RAM |
| 4 | Streaming Pattern | Робота з великими файлами та БД |
| 5 | Pipeline Pattern | Конвеєр трансформацій у стилі Unix pipes |

---

## Частина 1: Iterator Pattern — Стандарт перебору та внутрішня магія `for`

---

### Передбачення

> **Питання:** Усі ви щодня використовуєте цикл `for item in data:`.  
> Але чи знаєте ви, що в Python насправді **не існує циклу `for`** у класичному розумінні?  
> Що насправді викликає інтерпретатор **«під капотом»**?

### Пояснення: Протокол ітератора

Цикл `for` — це лише **синтаксичний цукор** над патерном Ітератор.

В архітектурі Python **Ітератор — це об'єкт, який видає значення по одному за раз**.  
Щоб об'єкт став ітератором, він має реалізувати строгий протокол з двох дандер-методів:

1. `__iter__(self)` — повертає сам об'єкт ітератора.
2. `__next__(self)` — обчислює і повертає наступне значення.  
   Якщо даних більше немає — генерує виняток `StopIteration`.

Цей протокол розв'язує руки архітектору: ви можете передати в цикл файл, мережевий сокет або алгоритм, і Python обробить їх однаково.

In [ ]:
data = ["a", "b", "c"]

# Те, що ви пишете:
for item in data:
    print(item)

print("---")

# Те, як це бачить і виконує архітектура Python:
iterator = iter(data)        # Викликає data.__iter__()
while True:
    try:
        item = next(iterator)    # Викликає iterator.__next__()
        print(item)
    except StopIteration:        # Сигнал завершення потоку
        break

### Власний клас-ітератор

Реалізуємо ітератор вручну, щоб побачити протокол зсередини.  
Клас `CountUp` видає цілі числа від `start` до `stop` включно.

In [ ]:
class CountUp:
    """Ітератор: рахує від start до stop включно."""

    def __init__(self, start: int, stop: int):
        self.current = start
        self.stop = stop

    def __iter__(self):
        # Повертаємо сам об'єкт — він є і ітерованим, і ітератором
        return self

    def __next__(self):
        if self.current > self.stop:
            raise StopIteration  # Сигнал: потік вичерпано
        value = self.current
        self.current += 1
        return value


# Цикл for «не знає», що всередині — клас, список чи файл
for n in CountUp(1, 5):
    print(n)  # 1, 2, 3, 4, 5

---

## Частина 2: Generator Pattern — Lazy Execution (Ліниві обчислення)

---

Створення власних класів-ітераторів вимагає багато шаблонного коду  
(управління станом в `__init__`, ручний виклик `StopIteration`).  
Тому Python пропонує **Генератори**.

**Генератор — це функція, яка вміє ставити своє виконання на паузу та відновлювати його.**

| Механізм | `return` | `yield` |
|----------|----------|---------|
| Стан функції | Знищується | Зберігається |
| Локальні змінні | Втрачаються | Заморожуються |
| Наступний виклик | Починає спочатку | Продовжує з точки зупинки |

Коли клієнт запитує `next()`, функція «розморожується», виконує роботу до наступного `yield` і знову засинає.

### Ментальне порівняння: List vs Generator

> **Питання:** Ви хочете обробити 100 мільйонів чисел.  
> Що станеться в обох випадках з точки зору використання RAM?

In [ ]:
import sys

def to_gb(x):
    return x / (1024 ** 3)


eager_list = [x for x in range(1_000_000_000)]

size_bytes = sys.getsizeof(eager_list)

print(f"List RAM:      {size_bytes:>12,} bytes | {to_gb(size_bytes):.4f} GB")


lazy_gen = (x for x in range(1_000_000_000))
gen_bytes = sys.getsizeof(lazy_gen)

print(f"Generator RAM: {gen_bytes:>12,} bytes | {to_gb(gen_bytes):.8f} GB")

In [ ]:
import sys

# Жадібні обчислення (Eager evaluation) — завантажує все в пам'ять одразу
eager_list = [x for x in range(1_000_000)]
print(f"List RAM:      {sys.getsizeof(eager_list):>12,} байт")



In [ ]:
# Ліниві обчислення (Lazy evaluation) — зберігає лише алгоритм
lazy_gen = (x for x in range(1_000_000))
print(f"Generator RAM: {sys.getsizeof(lazy_gen):>12,} байт")

**Висновок:**  
Генератор не зберігає дані в пам'яті. Він зберігає **лише стан і алгоритм (формулу)** того, як створити наступний елемент.  
Це основа оптимізації високонавантажених систем.

### Генераторна функція: синтаксис `yield`

Перетворимо `CountUp` на генераторну функцію — позбавляємось усього шаблонного коду.

In [ ]:
def count_up(start: int, stop: int):
    """Генераторна функція: еквівалент класу CountUp, але лаконічніший."""
    current = start
    while current <= stop:
        yield current          # Заморозити функцію, повернути значення
        current += 1           # Виконається після наступного next()


gen = count_up(1, 5)
print(type(gen))               # <class 'generator'>

print(next(gen))               # 1 — функція виконується до першого yield
print(next(gen))               # 2 — відновлюється з точки зупинки

# Решту значень ітеруємо циклом
for n in gen:
    print(n)                   # 3, 4, 5

---

## Частина 3: Infinite Stream Pattern — Нескінченні послідовності

---

Оскільки генератори створюють дані лише тоді, коли їх просять, вони можуть описувати  
**нескінченні структури даних** — що принципово неможливо зробити за допомогою списків.

Це ідеальний патерн для:
- безперервного потоку подій (event stream),
- пулінгу мережевих запитів,
- зчитування даних з датчиків у реальному часі.

In [ ]:
import time

def sensor_data_stream():
    """Нескінченний потік даних (Infinite Stream).
    
    while True зі списком зупинило б програму назавжди —
    з yield це абсолютно безпечно.
    """
    state = 0
    while True:                    # Нескінченний цикл — але ми контролюємо паузи
        yield state
        state += 1
        time.sleep(0.05)           # Імітуємо затримку датчика


stream = sensor_data_stream()

# Беремо лише перші 5 значень з нескінченного потоку
for _ in range(5):
    print(f"Показник датчика: {next(stream)}")

---

## Частина 4: Streaming Pattern — Робота з великими даними

---

Генератори — це **єдиний правильний спосіб** читання масивних файлів або таблиць баз даних.

Відкриваючи файл у Python, ви вже працюєте з ітератором, який витягує дані по одному рядку,  
не завантажуючи весь 10-гігабайтний лог в пам'ять.

In [ ]:
# Патерн Streaming: обробка файла на ходу без переповнення пам'яті

def read_large_log(file_path: str):
    """Генератор-стример: зчитує ERROR-рядки по одному без буферизації."""
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:          # Протокол ітерації! Читає 1 рядок за раз.
            if "ERROR" in line:
                yield line.strip()


# --- Демонстрація на тимчасовому файлі ---
import tempfile, os

log_data = """INFO: Server started
ERROR: Disk full
WARNING: High latency
ERROR: DB timeout
INFO: Request processed
"""

with tempfile.NamedTemporaryFile(mode="w", suffix=".log", delete=False, encoding="utf-8") as f:
    f.write(log_data)
    tmp_path = f.name

# Стримінг: у RAM одночасно лише 1 рядок
for error_line in read_large_log(tmp_path):
    print(error_line)

os.unlink(tmp_path)  # Прибираємо тимчасовий файл

---

## Частина 5: Pipeline Pattern — Chain of Transformations (Конвеєри)

---

### Передбачення

> **Питання:** Як розбити складний монолітний код на модульні, незалежні компоненти,  
> які фільтрують та трансформують потік даних?

Тут розкривається справжня архітектурна потужність.  
Ви можете з'єднувати генератори один з одним, як труби на заводі  
(в стилі Unix pipes: `tail -f | grep python | awk ...`).

Дані протікають через конвеєр **по одному елементу за раз**.  
Жоден проміжний список не створюється. Буферизація між кроками відсутня.

In [ ]:
# ============================================================
# PIPELINE PATTERN: конвеєр з трьох незалежних генераторів
# ============================================================

# 1. Джерело даних (Source) — генерує сировину
def get_logs():
    yield "INFO: Server started"
    yield "ERROR: Disk full"
    yield "WARNING: High latency"
    yield "ERROR: DB timeout"


# 2. Фільтр (Filter) — відкидає зайве
def filter_errors(stream):
    for line in stream:
        if "ERROR" in line:
            yield line


# 3. Трансформація (Map) — змінює форму даних
def extract_message(stream):
    for line in stream:
        # Розбиваємо "РІВЕНЬ: повідомлення" → ["РІВЕНЬ", "повідомлення"]
        parts = line.split(": ", maxsplit=1)
        yield parts[1] if len(parts) == 2 else line


# --- Збірка конвеєра (Pipeline Assembly) ---
# Жоден з генераторів ще НЕ виконався — вони лише описують структуру потоку
log_stream   = get_logs()
errors_only  = filter_errors(log_stream)
messages     = extract_message(errors_only)

# Виконання (Sink) — «споживач», який запускає весь конвеєр
print("Помилки з логу:")
for msg in messages:
    print(f"  • {msg}")

### Як це працює в пам'яті

```
Sink (for msg in messages)
  └─ запитує елемент у extract_message
       └─ запитує елемент у filter_errors
            └─ запитує елемент у get_logs
                 └─ видає: "ERROR: Disk full"
       └─ фільтрує → пропускає
  └─ трансформує → "Disk full"
  └─ виводить → очищується з RAM
```

Один елемент (`"ERROR: Disk full"`) генерується, миттєво проходить через усі фази  
фільтрації та форматування і **знищується**. Пам'ять залишається порожньою.  
Це еталон **Data Flow архітектури**.

---

## Частина 2b: Eager vs Lazy — Детальне Порівняння

---


### Стратегія обчислення: що, коли і скільки коштує

| Критерій | List comprehension (Eager) | Generator expression (Lazy) |
|----------|---------------------------|-----------------------------|
| Синтаксис | `[x**2 for x in ...]` | `(x**2 for x in ...)` |
| Виконання | Одразу, повністю | На вимогу, по одному |
| RAM | Пропорційна кількості елементів | Константна (~88 байт) |
| Швидкість (малі дані) | **Швидше** (оптимізовано на C) | Трохи повільніше (overhead yield) |
| Швидкість (великі дані) | Може впасти (OOM) | **Стабільна** |
| Повторне використання | Так (список) | Ні (генератор вичерпується) |

> **Правило вибору:** якщо вам потрібен результат кілька разів або потрібна індексація — список.  
> Якщо дані великі або ви проходите лише один раз — генератор.


In [ ]:
import sys

# ---- Синтаксис: дужки vs квадратні дужки ----
squares_list = [x ** 2 for x in range(10_000)]   # Eager: список у пам'яті
squares_gen  = (x ** 2 for x in range(10_000))   # Lazy:  лише об'єкт-генератор

print(f"List size:      {sys.getsizeof(squares_list):>10,} байт")
print(f"Generator size: {sys.getsizeof(squares_gen):>10,} байт")

# ---- Генераторний вираз як аргумент функції ----
# Коли генератор — єдиний аргумент, зайві дужки не потрібні
total  = sum(x ** 2 for x in range(1_000_000))   # Жодного проміжного списку!
print(f"\nСума квадратів 1..1M = {total:,}")

# next() з генераторного виразу
evens = (x for x in range(100) if x % 2 == 0)
print(next(evens))  # 0 — обчислено лише перше значення
print(next(evens))  # 2


---

## Частина 6: `yield` — Функції як Кінцеві Автомати

---


### Ментальна модель VCR: Пауза та Відновлення

У класичному програмуванні функція виконує `return` — і **вмирає**, знищуючи всі локальні змінні.  
`yield` — це протилежність `return`. Це **заморозка часу і простору всередині функції**.

> **Питання:** Що відбудеться, якщо викликати функцію з `yield`?  
> Чи виконається код всередині неї одразу?


In [ ]:
def vcr_function():
    print("▶️  Play: Запуск функції...")
    yield "Перший кадр"            # Зупинка #1: передаємо значення, заморожуємось

    print("▶️  Play: Продовжуємо з місця зупинки...")
    yield "Другий кадр"             # Зупинка #2

    print("⏹  Stop: Кінець функції.")


# КРОК 1: виклик функції НЕ виконує жодного рядка коду!
player = vcr_function()
print("Об'єкт створено:", player)  # Лише об'єкт-генератор

print("\n--- next() #1 ---")
frame = next(player)               # Тепер код виконується до першого yield
print("Отримано:", frame)

print("\n--- next() #2 ---")
frame = next(player)               # Продовжується з точки зупинки
print("Отримано:", frame)

print("\n--- next() #3 ---")
try:
    next(player)                   # Більше yield немає → StopIteration
except StopIteration:
    print("Генератор вичерпано.")


### Генератор як Кінцевий Автомат (State Machine)

Оскільки генератор зберігає локальні змінні між викликами `next()`,  
він є ідеальним інструментом для написання **кінцевих автоматів** без громіздких класів.

> **Питання:** У функції нижче немає глобальних змінних.  
> Як вона запам'ятає значення `state` між викликами?


In [ ]:
def traffic_light():
    """Кінцевий автомат: нескінченний цикл зі збереженням стану."""
    transitions = {"Червоний": "Зелений", "Зелений": "Жовтий", "Жовтий": "Червоний"}
    state = "Червоний"
    while True:
        yield state                     # Заморожуємо: state збережений!
        state = transitions[state]      # Виконується після наступного next()


light = traffic_light()
for _ in range(6):
    print(next(light))   # Червоний → Зелений → Жовтий → Червоний → ...


### Двосторонній рух: метод `.send()`

Більшість розробників думають, що генератори можуть лише *віддавати* дані назовні.  
Але `yield` — це вираз, який також **приймає** значення ззовні через метод `.send()`.

Функція, що використовує `yield` для прийому значень, називається **сопрограмою (coroutine)**.  
Саме на цій ідеї побудовано `asyncio` та `async/await`.

> **Пастка ініціалізації:** перед першим `.send(значення)` потрібно "прокрутити"  
> генератор до першого `yield` через `next()` або `send(None)`.  
> Інакше отримаємо `TypeError: can't send non-None value to a just-started generator`.


In [ ]:
def accumulator():
    """Сопрограма: приймає числа через send() і накопичує суму."""
    total = 0
    while True:
        value = yield total    # yield: віддає поточну суму І чекає нового числа
        if value is None:
            break
        total += value


acc = accumulator()
next(acc)            # Прокручуємо до першого yield (ініціалізація)

print(acc.send(10))  # Передаємо 10 → отримуємо поточну суму: 10
print(acc.send(25))  # Передаємо 25 → отримуємо: 35
print(acc.send(5))   # Передаємо  5 → отримуємо: 40


---

## Частина 7: Трансформатори потоку — `map`, `filter`, `reduce`

---


У парадигмі Data Flow ми не пишемо цикли зі складними умовами.  
Ми використовуємо стандартні **будівельні блоки функціонального програмування**:

| Функція | Аналогія | Що робить |
|---------|----------|----------|
| `map(func, iter)` | Трансформатор | Змінює форму кожного елемента |
| `filter(pred, iter)` | Фільтр/вентиль | Відкидає елементи, що не проходять перевірку |
| `reduce(func, iter)` | Агрегатор | "Стискає" потік у одне значення |

> У Python 3 `map` і `filter` повертають **ледачі ітератори**, а не списки —  
> тобто вони ідеально вписуються в конвеєри без зайвого копіювання в пам'ять.


In [ ]:
from functools import reduce

temperatures_f = [32, 68, 98.6, 212, -40]   # Температури у Фаренгейтах

# map: трансформація (F → C)
to_celsius = map(lambda f: round((f - 32) * 5 / 9, 1), temperatures_f)

# filter: відбір (тільки додатні значення)
positive_only = filter(lambda c: c > 0, to_celsius)

# reduce: агрегація (сума всіх значень)
total_heat = reduce(lambda acc, c: acc + c, positive_only)

print(f"Сума додатних температур (°C): {total_heat}")

# Еквівалентний конвеєр через генераторні вирази (більш «Pythonic»)
result = sum(
    round((f - 32) * 5 / 9, 1)
    for f in temperatures_f
    if (f - 32) * 5 / 9 > 0
)
print(f"Те саме через genexpr:          {result}")


---

## Частина 8: `itertools` — Алгебра Ітераторів

---


Модуль `itertools` — це стандартна бібліотека Python, натхненна функціональними мовами  
(Haskell, SML). Її інструменти реалізовані на C та є надшвидкими будівельними блоками конвеєрів.

### 8.1 Нескінченні ітератори

> **Питання:** Ми хочемо нескінченну послідовність чисел з кроком `0.5`.  
> `range()` впаде з `TypeError` — він не підтримує дроби. Що зробить `itertools.count`?  
> Як зупинити нескінченність?


In [ ]:
import itertools

# count: нескінченний лічильник (підтримує float)
infinite = itertools.count(start=10, step=2.5)
print("count:", list(itertools.islice(infinite, 5)))  # [10, 12.5, 15.0, 17.5, 20.0]

# cycle: нескінченне повторення послідовності
colors = itertools.cycle(["red", "green", "blue"])
print("cycle:", [next(colors) for _ in range(7)])

# repeat: один елемент N разів (або нескінченно)
print("repeat:", list(itertools.repeat(7, times=4)))


### 8.2 Злиття потоків: `chain` замість конкатенації

Початківці об'єднують колекції через `+`. Але:
- `a + b` створює **новий об'єкт у пам'яті** і вимагає однакових типів (`TypeError` для різних).
- `chain(a, b, c)` не створює нічого — лише ітератор, що **перемикається** між джерелами.


In [ ]:
import itertools

a = (1, 2, 3)      # Кортеж
b = [4, 5, 6]      # Список
c = 'abc'          # Рядок

# print(a + b)     # TypeError! Різні типи

# chain: duck typing — не важливо, що саме всередині
stream = itertools.chain(a, b, c)
print(list(stream))  # [1, 2, 3, 4, 5, 6, 'a', 'b', 'c']

# chain.from_iterable: розгортає список ітерованих об'єктів
nested = [[1, 2], [3, 4], [5, 6]]
flat = list(itertools.chain.from_iterable(nested))
print(flat)          # [1, 2, 3, 4, 5, 6]


### 8.3 Комбінаторика: прощання з вкладеними циклами

Два-три вкладені `for` — сигнал тривоги. `itertools` замінює їх елегантною математикою.

> **Питання:** Чим відрізняються `permutations` і `combinations`?


In [ ]:
import itertools

items = ['A', 'B', 'C']

# permutations: порядок ВАЖЛИВИЙ → ('A','B') ≠ ('B','A')
print("Permutations:")
print(list(itertools.permutations(items, 2)))

# combinations: порядок НЕ важливий → ('A','B') == ('B','A')
print("\nCombinations:")
print(list(itertools.combinations(items, 2)))

# product: Декартовий добуток — замінює вкладені for-цикли
print("\nProduct (замість for x in A: for y in B:):")
print(list(itertools.product('AB', repeat=2)))  # 'A','A' теж можливий


### 8.4 Функціональний конвеєр: `dropwhile` + `islice`

**Кейс:** Гігантський лог-файл. Потрібні перші 3 критичні помилки, пропустивши всі INFO на початку.

Замість `while` з прапорцями — два рядки `itertools`:


In [ ]:
import itertools

log_events = [
    "INFO: Server started",
    "INFO: Listening on :8080",
    "WARNING: High memory usage",
    "ERROR: Disk full",
    "ERROR: DB timeout",
    "INFO: Retrying...",
    "ERROR: Connection refused",
]

# dropwhile: пропускаємо елементи, ПОКИ умова True;
#            як тільки умова стає False — починаємо пропускати все далі
past_warnings = itertools.dropwhile(lambda e: "ERROR" not in e, log_events)

# islice: беремо рівно 3 перші критичні події
first_3_errors = itertools.islice(past_warnings, 3)

print("Перші 3 помилки:")
for err in first_3_errors:
    print(" •", err)


---

## Частина 9: Реальні Архітектурні Патерни

---


### Патерн ETL: Чистка та Валідація Даних з Датчиків

Замість одного монолітного класу — ланцюг невеликих генераторів,  
кожен з яких можна тестувати незалежно.


In [ ]:
# --- Симуляція "брудного" потоку з датчика ---
def sensor_stream():
    raw_data = [None, 98.6, 37.0, None, 101.3, 36.6, 212.0, None, 40.1]
    yield from raw_data   # yield from: делегує ітерацію вкладеного об'єкта

# --- Крок 1: очищення (filter) ---
def clean_nulls(stream):
    """Відкидаємо None-записи."""
    for value in stream:
        if value is not None:
            yield value

# --- Крок 2: нормалізація (map) ---
def normalize_to_celsius(stream):
    """Переводимо Фаренгейти в Цельсії (якщо > 50°, то це F)."""
    for value in stream:
        yield round((value - 32) * 5 / 9, 1) if value > 50 else value

# --- Крок 3: групування (batch) ---
def batch(stream, size: int):
    """Групуємо по N записів для масової вставки в БД."""
    chunk = []
    for item in stream:
        chunk.append(item)
        if len(chunk) == size:
            yield chunk
            chunk = []
    if chunk:              # Залишок
        yield chunk

# --- Збірка ETL-конвеєра ---
pipeline = batch(
    normalize_to_celsius(
        clean_nulls(
            sensor_stream()
        )
    ),
    size=2
)

print("Батчі для запису в БД:")
for batch_data in pipeline:
    print(" •", batch_data)  # Імітація INSERT INTO sensors VALUES ...


---

## Архітектурні висновки

| # | Патерн / Інструмент | Механізм | Для чого |
|---|---------------------|----------|----------|
| 1 | **Iterator Pattern** | `__iter__` + `__next__` | Стандартизує перебір будь-яких структур |
| 2 | **Generator Pattern** | `yield` | Ліниві обчислення, економія RAM |
| 3 | **Eager vs Lazy** | `[]` vs `()` | Вибір стратегії за розміром і частотою читань |
| 4 | **Infinite Stream** | `while True` + `yield` | Нескінченні послідовності (датчики, події) |
| 5 | **Streaming Pattern** | `for line in file` | Терабайти на 2 ГБ RAM |
| 6 | **Pipeline Pattern** | Ланцюги генераторів | Модульна обробка: source → filter → map → sink |
| 7 | **State Machine** | `yield` + локальний стан | Кінцеві автомати без класів |
| 8 | **Coroutine** | `.send()` | Двосторонній обмін; основа `async/await` |
| 9 | **map / filter / reduce** | Функціональні будівельні блоки | Декларативні трансформації потоку |
|10 | **itertools** | C-оптимізовані ітератори | `chain`, `count`, `combinations`, `dropwhile`… |

---

> **Ключова думка:**  
> Генератор — це не просто «малий список». Це **опис алгоритму**, що живе у часі.  
> Списки — це **знімок даних**. Генератори — це **потік даних**.  
> Вибирайте відповідно до задачі.
